In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import ipywidgets as widgets

## Δειγματοληψία με Διατήρηση Τιμής (Zero-Order Hold)

Η **Δειγματοληψία με Διατήρηση Τιμής (Zero-Order Hold - ZOH)** είναι μία από τις απλούστερες μεθόδους μετατροπής ενός δειγματοληπτημένου σήματος σε συνεχούς χρόνου. Αντί της χρήσης των συναρτήσεων παρεμβολής $\texttt{sinc}()$ της ιδανικής δειγματοληψίας, η βασική ιδέα είναι ότι κάθε δείγμα κρατείται σταθερό μέχρι να φτάσει το επόμενο δείγμα.

Αν έχουμε ένα δειγματοληπτημένο σήμα

$$x[n] = x(nT_s)$$

όπου $T_s$ είναι η περίοδος δειγματοληψίας και

$$f_s = \frac{1}{T_s}$$

η συχνότητα δειγματοληψίας, τότε η ZOH ανακατασκευή ορίζεται ως

$$x_{\text{ZOH}}(t) = x[n], \quad nT_s \le t < (n+1)T_s$$

Δηλαδή, η τιμή του δείγματος $x[n]$ παραμένει σταθερή για όλο το χρονικό διάστημα μέχρι το επόμενο δείγμα.

Η ZOH μπορεί να περιγραφεί ως άθροισμα μετατοπισμένων ορθογώνιων παλμών. Ορίζουμε τον παλμό

$$p(t) = \begin{cases}
1, & 0 < t < T_s \\
0, & \text{αλλού}
\end{cases}$$

Τότε η ανακατασκευή γράφεται ως

$$x_{\text{ZOH}}(t) = \sum_{n=-\infty}^{\infty} x(nT_s) p(t-nT_s)$$

Αυτό δείχνει ότι κάθε δείγμα $x[n]$ πολλαπλασιάζει έναν ορθογώνιο παλμό διάρκειας $T_s$.

Η κρουστική απόκριση της ZOH είναι ο ίδιος ο ορθογώνιος παλμός:

$$h_{\text{ZOH}}(t) = u(t) - u(t-T_s)$$

όπου $u(t)$ είναι η γνωστή μας βηματική συνάρτηση.

Η ZOH μπορεί να θεωρηθεί ως ένα γραμμικό χρονικά αμετάβλητο σύστημα που δέχεται ως είσοδο μια σειρά συναρτήσεων Δέλτα με βάρη τα δείγματα:

$$x_s(t) = \sum_{n=-\infty}^{\infty} x[n] \delta(t-nT_s)$$

και δίνει έξοδο

$$x_{\text{ZOH}}(t) = x_s(t) * h_{\text{ZOH}}(t)$$

Η απόκριση συχνότητας της ZOH είναι

$$H_{\text{ZOH}}(f) = T_s e^{-j\pi f T_s}\operatorname{sinc}(fT_s)$$

Η απόκριση πλάτους θα είναι

$$|H_{\text{ZOH}}(f)| = T_s |\operatorname{sinc}(fT_s)|$$

και η απόκριση φάσης θα είναι

$$\angle H_{\text{ZOH}}(f) = -\pi f T_s$$

με τον εκθετικό όρο να αντιστοιχεί σε χρονική καθυστέρηση, δηλαδή η ZOH εισάγει καθυστέρηση μισής περιόδου δειγματοληψίας.

Από τη μορφή της απόκρισης πλάτους, βλέπετε ότι οι υψηλότερες συχνότητες μέσα στη βασική ζώνη εξασθενούν περισσότερο από τις χαμηλές. Το φαινόμενο αυτό λέγεται **amplitude drop**.

Για παράδειγμα, όσο η συχνότητα πλησιάζει τη συχνότητα Nyquist, $f_N = \frac{f_s}{2}$, η απόκριση της ZOH μειώνεται. Άρα ένα ημίτονο υψηλής συχνότητας ανακατασκευάζεται με μικρότερο πλάτος από ένα ημίτονο χαμηλής συχνότητας. Ας το δούμε αυτό πιο συγκεκριμένα.

Έστω το σήμα συνεχούς χρόνου $x(t) = A \sin(2\pi f_0 t)$ και η δειγματοληπτημένη έκδοσή του, $x[n] = A \sin(2\pi f_0 nT_s)$.

Η ανακατασκευή του σήματος μέσω ZOH είναι

$$x_{\text{ZOH}}(t) = \sum_{n=-\infty}^{\infty}A \sin(2\pi f_0 nT_s)p(t-nT_s)$$

Το αποτέλεσμα είναι ένα "κλιμακωτό" σήμα. Δεν είναι πραγματικό ημίτονο, αλλά μία προσέγγισή του από τμηματικά σταθερές συναρτήσεις (όπως θα δείτε παρακάτω).

Για μικρή $T_s$, δηλαδή για μεγάλη $f_s$, τα βήματα είναι μικρά και η ZOH προσέγγιση μοιάζει αρκετά με το αρχικό σήμα συνεχούς χρόνου. Για μικρή $f_s$, τα βήματα είναι μεγάλα και η ανακατασκευή είναι φτωχή.

Ας οπτικοποιήσουμε αυτή τη διαδικασία!

In [2]:
# Ας κατασκευάσουμε το παραπάνω ημίτονο
def generate_signal(freq, duration=0.01, fs_high=100000):
    t = np.linspace(0, duration, int(fs_high * duration), endpoint=False)
    x = np.sin(2 * np.pi * freq * t)
    return t, x

Με χρήση της $\texttt{interp1d}$ της ``Python`` μπορούμε να κατασκευάσουμε τη ZOH εύκολα όπως παρακάτω:

In [3]:
# Συνάρτηση ανακατασκευής με Διατήρηση Τιμής (ZOH)
def zoh_reconstruct(xn, tn, t_recon):
    return interp1d(tn, xn, kind='previous', fill_value='extrapolate')(t_recon)

Ας φτιάξουμε μια συνάρτηση $\texttt{ZOH}$ που υλοποιεί τη δειγματοληψία με διατήρηση τιμής.

In [4]:
def ZOH(freq=800, fs=5000):
    t_cont, x_cont = generate_signal(freq)
    t_s = np.arange(0, 0.01, 1/fs)
    x_s = np.sin(2 * np.pi * freq * t_s)

    t_recon = t_cont
    x_zoh = zoh_reconstruct(x_s, t_s, t_recon)

    plt.figure(figsize=(10, 6))
    plt.plot(t_cont, x_cont, label='Αρχικό σήμα', linewidth=2)
    plt.stem(t_s, x_s, linefmt='k-', markerfmt='ko', basefmt=' ', label='Δείγματα')
    plt.plot(t_recon, x_zoh, label='Δειγματοληψία με Διατήρηση Τιμής (ZOH)', linestyle='--', color='red')
    plt.title(f'Δειγματοληψία με Διατήρηση Τιμής \n(Συχνοτήτα σήματος: {int(freq)} Hz, Συχνότητα δειγματοληψίας: {int(fs)} Hz)')
    plt.xlabel('Χρόνος (s)')
    plt.ylabel('Πλάτος')
    plt.ylim(-1.5, 1.5)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

Ας χρησιμοποιήσουμε τα widgets για να μπορούμε να παίζουμε με τις τιμές της συχνότητας του σήματος και της συχνότητας δειγματοληψίας.

In [ ]:
# Τρέξτε το πρόγραμμα με διαδραστικά widgets
widgets.interact(ZOH, freq=widgets.FloatSlider(value=800, min=100, max=1000, step=100),
                 fs=widgets.FloatSlider(value=5000, min=500, max=16000, step=500));

interactive(children=(FloatSlider(value=800.0, description='freq', max=1000.0, min=100.0, step=100.0), FloatSl…

Θέστε τη συχνότητα freq ίση με $500$ Hz, και "παίξτε" με τη συχνότητα δειγματοληψίας. Απαντήστε παρακάτω στο εξής ερώτημα: για ποιές τιμές της συχνότητας δειγματοληψίας fs το ανακατασκευασμένο σήμα (κόκκινο) αρχίζει να "ταιριάζει" καλά με το αρχικό (μπλε)?

### Απάντηση: 

Παρατηρούμε ότι για αρκετά μεγάλες συχνότητες δειγματοληψίας ($fs > 6000$ Hz) ταιριάζουν καλά τα δυο σήματα.

---
---